[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/three_point_pose/three_point_pose.ipynb)

# Pose from Three Points

This example reconstructs the rigid rotation and translation of an object from the measured positions of three markers attached to it. Matching one marker fixes the position, matching the line through two markers fixes the edge direction, and matching the plane through all three fixes the roll about that line.

In projective geometric algebra, joining markers with `&` forms the edge line `target[0] & target[1]` and the marker plane `target_line & target[2]` inside the same algebra as the points themselves. Because points, lines, and planes are all blades of the algebra, the motor carrying one normalized element onto another of the same grade is `(target / source).square_root()` in all three stages, and multiplying the three stage motors gives `roll * turn * translation` acting by `pose >> source`. This three-stage factorization follows Steven De Keninck's [ganja.js motor reconstruction](https://github.com/enkimute/ganja.js/blob/master/examples/example_chapter11_motor_reconstruction.html).

In matrix rigid-body notation, each marker triple $(\mathbf a, \mathbf b, \mathbf c)$ builds a $3 \times 3$ Gram–Schmidt triad $T = [\mathbf t_1\;\mathbf t_2\;\mathbf t_3]$ via $\mathbf t_1 = (\mathbf b - \mathbf a)/|\mathbf b - \mathbf a|$, $\mathbf t_3 = (\mathbf t_1 \times (\mathbf c - \mathbf a))/|\mathbf t_1 \times (\mathbf c - \mathbf a)|$, and $\mathbf t_2 = \mathbf t_3 \times \mathbf t_1$, after which the rotation $R = T' T^\mathsf{T}$ and the origin-dependent translation $\mathbf d = \mathbf a' - R\,\mathbf a$ transform a point separately as $\mathbf x \mapsto R\,\mathbf x + \mathbf d$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_states
from examples.geometry.three_point_pose import render

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Motor = ga.gatype.rotor()                           # Rotor+

## Markers and their measurements

Three non-collinear markers sit at known places on the rigid object, and a known screw motion carries them to their measured places to provide exact test data; the reconstruction receives only the two point triples.

In [ ]:
source = mv.point([[-0.8, -0.4, 0.0], [0.9, -0.3, 0.0], [-0.3, 0.8, 0.2]])      # [3] Point-
displacement = mv.xw * 0.7 + mv.yw * 0.2 + mv.zw * 0.4                          # [] Bivector+
rotation = mv.xy * 0.6 + mv.yz * 0.2                                            # [] Bivector+

In [ ]:
# Apply a known rigid motion to generate exact target measurements.
pose = displacement.exp() * rotation.exp()                                      # [] Motor+
target = pose >> source                                                         # [3] Point-
save_figure(render.pose_figure(source, target, source), "three_point_pose_measured");

The orange balls and dashed outline mark the measured target triple, while the translucent teal plate and solid edges show the moving object in its initial pose before alignment begins.

## Matching a point

The ratio of two unit-weight points is a pure translation along the line joining them by twice their separation, so its square root translates the first marker directly onto its measurement while leaving the object's orientation unchanged.

In [ ]:
# Two point reflections translate by twice the required displacement.
translation = (target[0] / source[0]).square_root()                             # [] Motor+
at_first_marker = translation >> source                                         # [3] Point-
save_figure(render.pose_figure(source, target, at_first_marker), "three_point_pose_point");

One corner of the teal triangle now coincides with its orange target ball, while the two triangles still point along different edges and lie in different planes.

## Matching a line

With the first marker fixed on its target, the edge line through the first two markers and the edge line through their measurements intersect at that shared point. The square root of the two normalized lines' ratio rotates one line onto the other in their common plane without disturbing the matched corner.

In [ ]:
# Turn the joining line while keeping the matched first point fixed.
target_line = (target[0] & target[1]).normalized()                              # [] Line-
current_line = (target[0] & at_first_marker[1]).normalized()                    # [] Line-
turn = (target_line / current_line).square_root()                               # [] Motor+
along_first_edge = turn >> at_first_marker                                      # [3] Point-
save_figure(render.pose_figure(source, target, along_first_edge), "three_point_pose_line");

Both endpoints of the first teal edge now sit on their orange markers, leaving only a single rotational degree of freedom around that shared edge line.

## Matching a plane

With the first edge aligned, the plane joining that edge to the third marker and the plane joining it to the third measurement hinge on the same line. The square root of the normalized planes' ratio rolls one plane onto the other about the hinge line, and multiplying the three stage motors gives the complete pose.

In [ ]:
# Only rotation about that line remains; matching the marker planes removes it.
target_plane = (target_line & target[2]).normalized()                           # [] Plane-
current_plane = (target_line & along_first_edge[2]).normalized()                # [] Plane-
roll = (target_plane / current_plane).square_root()                             # [] Motor+
reconstructed_pose = roll * turn * translation                                  # [] Motor+
aligned_markers = reconstructed_pose >> source                                  # [3] Point-
save_figure(render.pose_figure(source, target, aligned_markers), "three_point_pose_plane");

All three corners and edges of the teal plate now cover the dashed orange target triangle.

## The three motions in turn

Taking a fraction of each stage motor's bivector logarithm and exponentiating interpolates smoothly along that stage's screw axis from start to finish. Rendering the full three-stage screw animation on the GPU takes around ten seconds.

In [ ]:
n_steps = 39
frame_duration_ms = 40

In [ ]:
def alignments(source: Point, increments: Motor, fractions: np.ndarray) -> Iterator[Point]:
    """Yields the marker triple (`[..., 3] Point-`) along each incremental motor's screw in turn."""
    current = source
    yield current
    for increment in increments:
        start = current
        for fraction in fractions:
            # A fraction of the motor's logarithm moves at a constant rate along its screw.
            step_motor = (increment.log() * fraction).exp()                     # [...] Motor+
            current = step_motor >> start                                       # [..., 3] Point-
            yield current


fractions = np.linspace(0, 1, n_steps + 1)[1:]
# Ease in and out of each motion.
fractions = fractions * fractions * (3 - 2 * fractions)
increments = stack([translation, turn, roll])                                   # [3] Motor+
states = show_states(
    alignments(source, increments, fractions),
    render.pose_animation, source, target,
    name="three_point_pose", duration_ms=frame_duration_ms,
);

The teal plate translates to the first orange marker, pivots around that corner until the first edge aligns, and rolls around the aligned edge onto the target plane.

In [ ]:
# checks
assert (at_first_marker[0] & target[0]).norm() < 1e-10
assert ((along_first_edge[:2] & target_line).norm() < 1e-10).all()
assert (reconstructed_pose * ~pose - 1).scalar_norm_squared() < 1e-20
assert ((aligned_markers & target).norm() < 1e-10).all()
assert ((states[-1] & target).norm() < 1e-10).all()